# C++ 第 4 章：资源管理与异常

> **开始前：选择 C++17 内核。** 本章不能使用 C17 内核。在线实验使用固定数据或 `std::istringstream`，键盘输入 `std::cin` / `std::getline(std::cin, ...)` 放在本地完整程序中练习。

**先修**：C 部分的变量、控制语句、数组、指针和函数；按 C++ 章节顺序学习。

**本章目标**：理解 RAII、独占所有权与移动；用具体异常报告失败。

**学习方法**：先预测，再运行、修改并解释。块内局部变量可重复运行；修改函数、类或模板后，使用 Kernel → Restart Kernel and Run All Cells。各章互不依赖内核状态。

**保存作业**：见 [课程入口](https://github.com/luckymark/c_course#保存作业与继续学习)。

## 1. 优先使用自动管理资源的对象

C 中 malloc/free 配对；C++ 中 string/vector 管理自己的存储，文件流管理文件句柄，这种“资源生命周期跟随对象”的方式称为 RAII。一般集合用 vector，不要为普通数组手写 new[]/delete[]。raw pointer 或引用通常只表示观察，必须确保被观察对象仍存活。

对象离开作用域，包括异常退出时，已构造局部对象会析构。析构函数应负责清理，不向外抛异常。

In [ ]:
#include <iostream>
#include <vector>
{
    const std::vector<int> original{85, 90};
    auto copy = original;
    copy.push_back(60);
    std::cout << original.size() << ' ' << copy.size() << '\n';
}

## 2. 需要动态对象时，用 unique_ptr 表示唯一所有者

`make_unique<T>` 创建对象并交给 unique_ptr 管理。unique_ptr 不可复制，可以移动；移动后源 unique_ptr 为空，目标持有对象。`std::move` 本身是转换表达式，实际资源转移由移动构造/赋值实现。不要将“移动后一定为空”推广到所有类型：例如 moved-from string 只保证有效但状态未指定。

共享所有权才考虑 shared_ptr，它有成本，也要防止循环引用；此处不需要它。

In [ ]:
#include <memory>
#include <utility>
#include <iostream>
{
    auto owner = std::make_unique<int>(85);
    auto next = std::move(owner);
    std::cout << (owner == nullptr) << ' ' << *next << '\n';
}

## 3. 明确区分正常缺失与错误

输入可预期的失败可以返回状态；违反约束的操作可抛具体异常。这里 at 越界抛 out_of_range，在能处理的边界捕获，不静默忽略；不要通过 [] 实际演示未定义行为。

**预测**：捕获后会不会执行最后一行？

In [ ]:
#include <vector>
#include <stdexcept>
#include <iostream>
{
    const std::vector<int> scores{85};
    try { std::cout << scores.at(1) << '\n'; }
    catch (const std::out_of_range&) { std::cout << "index out of range\n"; }
    std::cout << "continued\n";
}

## 小练习：建立独占对象并返回其值

只填写 BEGIN ANSWER 与 END ANSWER 之间的代码。先预测，再运行；初始 FAIL 表示尚未作答。完成后应全部 PASS。

In [ ]:
#include <memory>
#include <iostream>
{
    int result = -1;
    /* BEGIN ANSWER */
    // 用 make_unique 创建值为42的 int，并读取它。
    /* END ANSWER */
    std::cout << (result == 42 ? "PASS" : "FAIL") << '\n';
}

<details><summary>提示</summary>

先检查空数据和边界，再处理一般情况；只计算题目需要的返回值，不改变输入。

</details>

<details><summary>参考答案：完成后再展开</summary>

```cpp
const auto value = std::make_unique<int>(42);
result = *value;
```

</details>

## 本地实践：完整 C++17 程序

把代码保存为 `lesson.cpp`，在终端执行 `c++ -std=c++17 -Wall -Wextra -Wpedantic lesson.cpp -o lesson`，编译成功后运行 `./lesson`。Windows GCC 使用 `g++` 和 `-o lesson.exe`，PowerShell 运行 `.\lesson.exe`。完整程序不要直接放入在线单元或手动调用 main。

```cpp
#include <iostream>
#include <sstream>
#include <stdexcept>
#include <string>
int parse_score(const std::string& text) {
    std::istringstream input{text};
    int value{};
    std::string extra;
    if (!(input >> value) || (input >> extra)) { throw std::invalid_argument("invalid text"); }
    if (value < 0 || value > 100) { throw std::out_of_range("invalid score"); }
    return value;
}
int main() {
    std::string line;
    if (!std::getline(std::cin, line)) { std::cout << "missing input\n"; return 1; }
    try { const int score = parse_score(line); std::cout << score << '\n'; }
    catch (const std::invalid_argument& error) { std::cout << error.what() << '\n'; return 1; }
    catch (const std::out_of_range& error) { std::cout << error.what() << '\n'; return 1; }
}
```

验收输入与输出（`\n` 表示换行）：

- 输入 `"60\n"` → 输出 `"60\n"`，退出码 0。
- 输入 `"x\n"` → 输出 `"invalid text\n"`，退出码 1。
- 输入 `"101\n"` → 输出 `"invalid score\n"`，退出码 1。
- 输入 `""` → 输出 `"missing input\n"`，退出码 1。

## 分层练习与自查

1. **基础**：在纸上画出 unique_ptr 移动前后的所有权。
2. **改错**：`auto second = owner;` 为什么不能复制 unique_ptr？不要改用裸指针规避所有权问题。
3. **应用**：给本地解析程序增加空白行、负数、超大数字和后缀字符的验收。
4. **选做**：本地用 ifstream 读取一行成绩，检查文件打开和读取失败；解释为什么没有手写 close 仍会释放句柄。